### Project: GL-MOVE-Engine

**Designation:** Global-Local Market Integrity & Volatility Engine
**Track:** Machine Learning — Unsupervised Learning
**Status:** Milestone Project

---

### I. Comprehensive Project Overview

The **Global-Local Market Integrity & Volatility Engine (GL-MOVE)** is a high-dimensional analytical framework designed to decompose multi-country financial return streams into latent factors. In quantitative finance, raw price action is a noisy mixture of global macroeconomic trends, regional sector movements, and idiosyncratic company-specific events.

This engine utilizes **Unsupervised Learning** to perform "Blind Source Separation." By applying **Principal Component Analysis (PCA)**, we extract the systemic "Market Factor" explaining the primary variance across international exchanges (e.g., PSE, NYSE, HKEX). We then utilize **Independent Component Analysis (ICA)** to isolate non-Gaussian signals that represent statistically independent shocks—essential for identifying "hidden" risks. Finally, the system integrates **Anomaly Detection** (Isolation Forest & Z-Score) to flag instances where local market behavior deviates from the expected latent structure, serving as a proxy for identifying market manipulation or flash crashes.

---

### II. The Technical Stack

* **Data Tier:** `yfinance` API (Price Data), `BeautifulSoup4`/`Selenium` (Metadata Scraping).
* **Core Logic:** `scikit-learn` (PCA, FastICA, Isolation Forest), `pandas`, `numpy`, `statsmodels`.
* **Backend:** `FastAPI` (Asynchronous inference and data serving).
* **Frontend:** `React.js` with `D3.js` or `Recharts` (High-fidelity signal visualization).
* **Business Intelligence:** `Power BI` (Historical time-series volatility and risk ledger via SQL/CSV export).

---

### III. Folder Structure Validation & Optimization

#### Critique of User-Proposed Structure:

Your proposed structure (`src/model`, `notebooks`, `pipelines`, `data`, `ingestion`, `scraper`, `api/`, `frontend/`) is a solid start but contains architectural redundancies.

1. **Redundancy:** `ingestion` and `scraper` should be modules within a unified data package to ensure modularity.
2. **Ambiguity:** `data` should remain a root-level directory for storage, not logic.
3. **Scalability:** `pipelines` should explicitly separate training pipelines from inference pipelines.

#### The Architect’s Blueprint:

```text
ML/02-Unsupervised-Learning/GL-MOVE-Engine/
├── api/                        # FastAPI application logic
├── frontend/                   # React.js application
├── data/                       # Local storage (raw/, processed/, metadata/)
├── notebooks/                  # R&D and exploratory data analysis
├── artifacts/                  # Saved model weights (.pkl, .joblib)
├── pbix/                       # Power BI project files and templates
└── src/                        # Core Package Logic
    ├── __init__.py
    ├── components/             # Individual ML modules (pca, ica, anomaly)
    ├── data/                   # Data logic (ingestion.py, scraper.py, loaders.py)
    ├── pipelines/              # Orchestration (train_pipeline.py, inference_pipeline.py)
    └── utils/                  # Helper functions (math_ops.py, log_utils.py)

```

---

### IV. Revised Project Phases

| Phase | Designation | Objective | Implementation |
| --- | --- | --- | --- |
| **I** | **Hybrid Ingestion** | Acquire multi-country OHLCV data + Scraped Sector Metadata. | `src/data/` |
| **II** | **Signal Engineering** | Log-return transformation and stationarity validation. | `src/utils/` |
| **III** | **Decomposition Training** | Fit PCA (Systemic) and ICA (Independent) factor models. | `src/components/` |
| **IV** | **Integrity Surveillance** | Training Isolation Forest on latent space for anomaly detection. | `src/components/` |
| **V** | **Inference API** | Deployment of FastAPI endpoints for real-time decomposition. | `api/` |
| **VI** | **Visual Intelligence** | Development of React Interface and Power BI Time-Series. | `frontend/` & `pbix/` |

---

### V. Mathematical Foundation: Phase I & II

To begin, we must ensure the data is prepared for linear decomposition. PCA assumes that the data is centered and that the relationships are captured by variance.

#### 1. Log-Returns and Stationarity

We transform prices $P$ into log-returns $r$ to ensure time-additivity and stationarity:


$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

#### 2. Standardization (Z-Score)

Because PCA is sensitive to scale, we must standardize each asset's returns:


$$z = \frac{r - \mu}{\sigma}$$


where $\mu$ is the mean return and $\sigma$ is the standard deviation. This ensures that a stock with high nominal volatility does not disproportionately bias the "Global Factor."
___

## Literature Study: Theoretical Foundations of the GL-MOVE-Engine

The **Global-Local Market Integrity & Volatility Engine (GL-MOVE)** is architected to address specific mathematical and structural failures identified in contemporary quantitative research. This study maps academic problem statements to the technical solutions integrated into the engine’s architecture.

---

### I. Literature Mapping: Deep Dive into Problems & Solutions

| Research Paper | Identified Problem (The "Why") | GL-MOVE Technical Solution (The "How") |
| --- | --- | --- |
| **A Non-Normal Principal Components Model for Security Returns** | **The Gaussian Fallacy:** Standard Factor Models (BARRA, CAPM) rely on the covariance matrix $\Sigma$, assuming data is governed by mean and variance. Real-world returns exhibit **Excess Kurtosis** ($K > 3$), meaning "Black Swan" events are more frequent than a normal distribution predicts. Standard PCA "smears" these extreme events across multiple components. | **Residual Decomposition Pipeline:** We implement PCA to extract the "Market Beta" ($PC_1$, capturing $\approx 40-60\%$ of variance). Instead of discarding residuals as "noise," they are passed to a non-linear processing stage. This ensures "fat-tail" information is preserved for high-fidelity risk analysis. |
| **Blind Source Separation of Audio Signals using ICA and Wavelets** | **The Multi-Source Mixing Problem:** Financial markets suffer from the "Cocktail Party Effect." An observed price change at time $t$ is a linear mixture of global macro-trends, regional sector shocks, and company-specific news. These sources are often non-Gaussian and statistically independent; PCA cannot separate them as it only seeks orthogonal axes. | **Blind Source Separation (BSS):** By utilizing **FastICA**, the engine maximizes **Negentropy** to "unmix" the return stream into statistically independent market shocks. This allows for the isolation of specific risks (e.g., oil price shocks vs. currency devaluation) that appear blurred in variance-only models. |
| **Anomaly Detection as Integrity Surveillance** | **Dynamic Threshold Failure:** Traditional surveillance relies on fixed Z-Score thresholds on raw price action. During regime shifts, a $3\sigma$ move might be "normal," leading to high false positives. Conversely, sophisticated manipulation often stays within price thresholds while breaking the **latent correlation structure**. | **Latent Space Surveillance:** We monitor the **Isolation Forest Path Length** within the ICA factor space rather than price space. If an Independent Component shows an anomaly while the Global Factor remains stable, it indicates a structural breach (e.g., localized manipulation) that price-only models miss. |

---

### II. Mathematical Foundation: Signal Engineering

For the **Blind Source Separation** logic to hold, the input matrix $X$ must satisfy the condition of **Stationarity**.

#### 1. Log-Return Transformation

We transform raw prices $P$ into log-returns $r$ to achieve time-additivity and stationarity:


$$r_t = \ln(P_t) - \ln(P_{t-1})$$

#### 2. The Augmented Dickey-Fuller (ADF) Constraint

We perform a unit root test to verify stationarity. The model tests the null hypothesis $H_0$ that the series is non-stationary:


$$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \epsilon_t$$


A p-value $p < 0.05$ is required for an asset to be deemed "safe" for the decomposition pipeline.

#### 3. Negentropy Maximization (ICA)

Unlike PCA, which maximizes variance, ICA maximizes **Negentropy** $J(y)$, a measure of non-Gaussianity:


$$J(y) \approx c [E\{G(y)\} - E\{G(\nu)\}]^2$$


Where $G$ is a non-quadratic contrast function and $\nu$ is a Gaussian variable.

---

### III. Links & References

1. **A Non-Normal Principal Components Model for Security Returns**
* **Focus:** Financial Factor Modeling & PCA Limitations.
* [Access via SSRN](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3352062)


2. **Blind Source Separation of Audio Signals Using ICA and Wavelets**
* **Focus:** The Mathematical Foundation of Independent Component Analysis.
* [Access via ResearchGate](https://www.researchgate.net/publication/221632592_Blind_Source_Separation_of_audio_signals_using_independent_component_analysis_and_wavelets)


3. **Anomaly Detection as Integrity Surveillance**
* **Focus:** Real-time monitoring and ML-based security in financial data.
* [Access via IJSRA (PDF)](https://ijsra.net/sites/default/files/fulltext_pdf/IJSRA-2025-1824.pdf)